In [9]:
import os, io, sys
from pathlib import Path
import contextlib
from joblib import parallel_backend
from functools import wraps
from itertools import product

import numpy as np
import pandas as pd
import sympy as sp

import torch
from TorchSisso import SissoModel
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.utils.validation import check_is_fitted
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import ParameterGrid, LeaveOneOut
from sklearn.neighbors import NearestNeighbors
from sympy import symbols

import time, datetime
from collections import Counter

import utils

# Settings

## Data Loading and SISSO hyperparameter settings

In [2]:
# Unit Converted to SI unit for mathematical operation
name = "CrTeNW_data_UC" 

root_dir = str(Path(os.getcwd()))
from_dir = root_dir + "/data/"
to_dir = root_dir + "/results/"

df_cleaned, X, Y, clean_feature_list, clean_result_col = utils.load_and_clean_data(name, feature_col_num=0,target="length")


Loading CrTeNW_data_UC dataset...
Features list: ['Te', 'Size', 'Ar', 'H_2', 'R_temp', 'G_temp', 'G_time', 'Q_Pre']
Feature size: 8
Dataset size: 94
Target column: Length
Sum of target values: 0.00073408
Mean of target values: 7.80936170212766e-06
Y non-zero: 45 
Y zero: 49


In [3]:
# Switching to the TorchSisso format (1st column as target, then the rest are features)
df_X = pd.DataFrame(X, columns=clean_feature_list)
df_Y = pd.Series(Y,   name   =clean_result_col)

df_model = pd.concat([df_Y, df_X], axis=1)
df_model = df_model[[clean_result_col] + clean_feature_list]

X = df_model[clean_feature_list].values
y = df_model[clean_result_col].values
n_samples, n_feats = X.shape

# ----- unit list & symbols -----
uL, uT, uTh, uM = symbols('uL uT uTh uM')
dimensionality = [
    'uL',                # Te
    'uL',                # Size
    'uL**3*uT**-1',      # Ar
    'uL**3*uT**-1',      # H_2
    'uTh',               # R_temp
    'uTh',               # G_temp
    'uT',                # G_time
    'uM'                 # Q_Pre
]

L0 = df_model['Length'].mean()           
df_model['Length*'] = df_model['Length'] / L0

#output_dim = 1                           # dimension-less
#output_dim = uT**-1                      # per-second
output_dim = uT                           # second
clean_result_col = 'Length*'              # new target = dimensionless
df_model = df_model[[clean_result_col] + clean_feature_list]

print("df_model:\n", df_model)

df_model:
      Length*     Te   Size        Ar           H_2  R_temp  G_temp  G_time  \
0   0.000000  0.010  0.025  0.000002  1.170000e-07   983.0   983.0   180.0   
1   0.000000  0.005  0.025  0.000002  1.170000e-07   983.0   983.0   180.0   
2   0.000000  0.005  0.025  0.000002  1.170000e-07   983.0   983.0   180.0   
3   0.000000  0.016  0.025  0.000002  1.170000e-07   983.0   983.0   180.0   
4   0.000000  0.010  0.025  0.000002  1.670000e-07   983.0   983.0   180.0   
..       ...    ...    ...       ...           ...     ...     ...     ...   
89  2.138459  0.012  0.031  0.000003  6.000000e-07  1020.0  1020.0   180.0   
90  2.599444  0.012  0.031  0.000003  2.000000e-07  1020.0  1020.0   180.0   
91  2.061628  0.012  0.031  0.000003  2.000000e-07  1020.0  1020.0   300.0   
92  0.000000  0.012  0.031  0.000003  2.000000e-07   983.0   983.0   180.0   
93  0.000000  0.012  0.031  0.000003  3.000000e-07   983.0   983.0   180.0   

       Q_Pre  
0   0.000003  
1   0.000003  
2   0.0

In [4]:
# SISSO hyperparameter settings (operators, rung, ...)
basic_ops = ('+','-','*','/','exp','pow(-1)')
ops1 = ('+','-','*','/','exp','ln','pow(2)','pow(1/2)','pow(-1)')
ops2 = ('+','-','*','/','exp','ln','pow(2)','pow(3)','pow(1/2)','pow(1/3)', 'pow(-1)')
grid = {
    "n_expansion": [3],
    "n_term":      [1, 2, 3],
    "k":           [10, 20, 30],
    "operators":   [basic_ops, ops1, ops2],
    "use_gpu": [True]
}
print("Shaping dimension:", output_dim)
print("Total grid points:", len(list(ParameterGrid(grid))))

Shaping dimension: uT
Total grid points: 27


## Patching the original library for bug correction and extensive use

In [5]:
# ----------------------------------------------------------------------
# Wrap torch.tensor: convert ANY pandas Series to NumPy first
# ----------------------------------------------------------------------
@contextlib.contextmanager
def pandas_safe_tensors():
    if hasattr(torch, "_original_tensor"):
        yield
        return

    orig = torch.tensor                           
    def _tensor(data, *args, **kwargs):
        if isinstance(data, pd.Series):
            data = data.to_numpy()
        return orig(data, *args, **kwargs)

    torch.tensor = _tensor
    torch._original_tensor = orig
    try:
        yield                                      # << run SISSO here
    finally:
        torch.tensor = orig                        # restore
        del torch._original_tensor

print("[DEBUG] torch.tensor patched once?  ",hasattr(torch, "_original_tensor"),"   wrapper name =", torch.tensor.__name__)
#--------------------------------------------------------------------#
# Ignore 4th return value (final_eq)
#--------------------------------------------------------------------#
from TorchSisso.Regressor_dimension import Regressor as _Regressor
from TorchSisso.model import SissoModel as _SM

# Save the original method
_orig_fit  = _Regressor.regressor_fit
_orig_eval = _SM.evaluate

# Drops the 4th return value
def _patched_fit(self, *args, **kwargs):
    out = _orig_fit(self, *args, **kwargs)
    return out[:3]          # keep rmse, equation, r2

def _patched_eval(self, equation, df):
    return _orig_eval(self, equation.replace('^', '**'), df)

_Regressor.regressor_fit = _patched_fit
_SM.evaluate = _patched_eval

print("[DEBUG] SissoModel.evaluate patched?  caret handled =>", _SM.evaluate is _patched_eval)
#--------------------------------------------------------------------#
# PATCH: Safe transcendental operators for fixing output dimension
#--------------------------------------------------------------------#
from TorchSisso.DimensionalFeatureSpaceConstruction import (
        feature_space_construction as _FSC)
from sympy import symbols
import torch, numpy as np

_DIMLESS = symbols('1')
_BAD_OPS = {'exp', 'ln', 'log'}

_orig_dim2non = _FSC.dimension_to_non_dimension_feature_expansion

def _safe_dim2non(self):
    """
    1) Temporarily drop exp/ln/log while any dimensional group is being turned into non-dimensional space.
    2) Run the library original routine (with the pruned list).
    3) Append exp/ln/log only for the columns whose stored unit tag is already dimensionless.
    """
    old_ops = list(self.operators)
    self.operators = [op for op in old_ops if op not in _BAD_OPS]

    # original work 
    vals, names, units = _orig_dim2non(self)

    # restore full list
    self.operators = old_ops

    # add transcendental transforms of unit-1 columns
    dimless_idx  = [i for i, u in enumerate(units) if u == _DIMLESS]
    if dimless_idx:
        base_vals  = vals[:, dimless_idx]
        base_names = [names[i] for i in dimless_idx]

        for op in _BAD_OPS & set(old_ops):
            if op == 'exp':
                extra_vals  = torch.exp(base_vals)
                extra_names = [f'(exp({n}))' for n in base_names]
            elif op == 'ln':
                extra_vals  = torch.log(base_vals)
                extra_names = [f'(ln({n}))'  for n in base_names]
            elif op == 'log':
                extra_vals  = torch.log10(base_vals)
                extra_names = [f'(log({n}))' for n in base_names]

            vals   = torch.cat((vals, extra_vals), dim=1)
            names += extra_names
            units += [_DIMLESS] * extra_vals.shape[1]

    # final NaN / Inf filtering (same logic as upstream)
    bad = torch.any(torch.isnan(vals) | torch.isinf(vals), dim=0)
    vals  = vals[:, ~bad]
    names = [n for n, keep in zip(names, ~bad) if keep]
    units = [u for u, keep in zip(units, ~bad) if keep]

    # Remove log(exp(..)) and exp(ln(..)) duplicates
    def _is_tautology(name):
        return name.startswith('(log(exp(') or name.startswith('(exp(ln(')
    keep_mask = [not _is_tautology(n) for n in names]
    vals  = vals[:, keep_mask]
    names = [n for n,k in zip(names, keep_mask) if k]
    units = [u for u,k in zip(units, keep_mask) if k]

    return vals, names, units

_FSC.dimension_to_non_dimension_feature_expansion = _safe_dim2non
print("[PATCH] exp/ln/log now allowed *only* on existing unit-1 features")
#--------------------------------------------------------------------#

[DEBUG] torch.tensor patched once?   False    wrapper name = tensor
[DEBUG] SissoModel.evaluate patched?  caret handled => True
[PATCH] exp/ln/log now allowed *only* on existing unit-1 features


## Helper function

In [6]:
@contextlib.contextmanager
def silence_torchsisso():
    """Redirect stdout/stderr to a dummy buffer inside the with-block."""
    _null = io.StringIO()
    _old_out, _old_err = sys.stdout, sys.stderr
    try:
        sys.stdout, sys.stderr = _null, _null
        yield
    finally:
        sys.stdout, sys.stderr = _old_out, _old_err

def knn_weights(X_, k=7, power=1.0):
    """Return w-mean distance to k-NNs  (larger w ==> sparser point)."""
    nbrs   = NearestNeighbors(n_neighbors=k + 1).fit(X_)
    d_k    = nbrs.kneighbors(X_)[0][:, 1:].mean(axis=1)   # skip self (col 0)
    w      = d_k ** power
    return w / w.mean()           # normalise  ⟨w⟩ = 1

def apply_weights(df, s_vec):
    """Multiply target and all features row-wise by s_vec (shape = N×1)."""
    df_w        = df.copy()
    df_w.iloc[:, 0] = df_w.iloc[:, 0] * s_vec      
    df_w.iloc[:, 1:] = df_w.iloc[:, 1:].mul(s_vec, axis=0)
    return df_w

def scale_features_only(df, s_vec):
    """Multiply only the feature columns (col 1:) by s_vec."""
    df_w        = df.copy()
    df_w.iloc[:, 1:] = df_w.iloc[:, 1:].mul(s_vec, axis=0)
    return df_w

def loo_score(params):
    loo = LeaveOneOut()
    y_true, y_pred = [], []
    tic = time.perf_counter()
    print("Grid combo →", params)

    for train_idx, test_idx in loo.split(X):
        X_std = (X[train_idx] - X[train_idx].mean(axis=0)) / X[train_idx].std(axis=0, ddof=0)
        # K-NN inverse-density weights for the training split
        w_tr       = knn_weights(X_std, k=7, power=1.0)
        sqrt_w_tr  = np.sqrt(w_tr)
        X_test_std = (X[test_idx] - X[train_idx].mean(axis=0)) / X[train_idx].std(axis=0, ddof=0)
        s_test = np.sqrt(knn_weights(np.vstack([X_std, X_test_std]), k=7, power=1.0)[-1])

        df_train_w = apply_weights(df_model.iloc[train_idx], sqrt_w_tr)
        df_test_w  = scale_features_only(df_model.iloc[test_idx], s_test)

        # SISSO fit (same as before)
        with pandas_safe_tensors():
            sm = SissoModel(
                data              = df_train_w.astype('float64'),
                operators         = params.get("operators",
                                            ('+','-','*','/','exp','exp(-1)')),
                n_expansion       = params["n_expansion"],
                n_term            = params["n_term"],
                k                 = params["k"],
                dimensionality    = dimensionality,
                output_dim        = output_dim,
                initial_screening = params.get("initial_screening"),
                use_gpu           = params["use_gpu"],
            )
        
            with silence_torchsisso():
                rmse_fit, eqn, _ = sm.fit()       

        # Evaluate: scale back the prediction 
        y_hat_w, _ = sm.evaluate(eqn, df_test_w)
        if y_hat_w is None or not np.isfinite(y_hat_w).all():
            raise ValueError("prediction nan/inf")
        y_pred.append(float(y_hat_w / s_test))
        y_true.append(df_model.iloc[test_idx, 0])

    rmse_cv = np.sqrt(mean_squared_error(y_true, y_pred))
    r2_cv   = r2_score(y_true, y_pred)

    # Refit on data with consistent weights 
    X_full_std  = (X - X.mean(axis=0)) / X.std(axis=0, ddof=0)
    w_full      = knn_weights(X_full_std, k=7, power=1.0)
    sqrt_w_full = np.sqrt(w_full)
    with pandas_safe_tensors():  
        sm_full = SissoModel(
            data              = apply_weights(df_model, sqrt_w_full).astype('float64'),
            operators         = params["operators"],
            n_expansion       = params["n_expansion"],
            n_term            = params["n_term"],
            k                 = params["k"],
            dimensionality    = dimensionality,
            output_dim        = output_dim,
            initial_screening = params.get("initial_screening"),
            use_gpu           = params["use_gpu"],
        )
        
        with silence_torchsisso():
            rmse_full_w, eqn_full, _ = sm_full.fit()   # weighted metrics

    # Diagnostics: ONCE per param set
    try:
        ops_equal = tuple(getattr(sm_full, "operators", ())) == params["operators"]
        k_val     = getattr(sm_full, "k", None)        # SissoModel does not expose k
        k_equal   = (k_val == params["k"]) if k_val is not None else "n/a"
        coef_attr = getattr(sm_full, "coefs_", getattr(sm_full, "coef_", None))
        has_nan   = coef_attr is not None and np.isnan(coef_attr).any()
        print("Ops_match:", ops_equal," k_match:", k_equal," NaN_coef:", has_nan)
    except Exception as e:
        print("Diagnostic skipped →", e)

    # Evaluate the final equation on original scale
    y_hat_full_w, _ = sm_full.evaluate(
        eqn_full, scale_features_only(df_model, sqrt_w_full)
    )
    y_hat_full = y_hat_full_w / sqrt_w_full  
    rmse_full  = np.sqrt(mean_squared_error(df_model.iloc[:, 0], y_hat_full))
    r2_full    = r2_score(df_model.iloc[:, 0], y_hat_full)

    # Time and output
    elapsed = time.perf_counter() - tic
    print(f"Finished:  RMSE_cv={rmse_cv:6.3f} "
          f"R2_cv={r2_cv:5.3f}  R2_full={r2_full:5.3f} "
          f"(elapsed {elapsed:6.2f} s)")

    return rmse_cv, r2_cv, eqn_full, rmse_full, r2_full


# Leave-One-Out-Cross-Validation (LOOCV) for hyperparameter tuning

In [7]:
results = []
best_by_rmse = {"rmse_cv": np.inf}
best_by_r2   = {"r2_cv":   -np.inf}

# Grid Search
t_start = time.perf_counter()

for vals in product(*grid.values()):
    params = dict(zip(grid.keys(), vals))
    try:
        rmse_cv, r2_cv, eqn, rmse_f, r2_f = loo_score(params)
    except Exception as e:
        print("skip", params, "→", e)
        continue

    record = {**params,
              "rmse_cv":  rmse_cv,  "r2_cv":  r2_cv,
              "rmse_full": rmse_f,  "r2_full": r2_f,
              "equation": eqn}
    results.append(record)

    if rmse_cv < best_by_rmse["rmse_cv"]:
        best_by_rmse = record
    if r2_cv > best_by_r2["r2_cv"]:
        best_by_r2 = record

df_results = pd.DataFrame(results)

# Worse Case Scenario
if df_results.empty:
    raise RuntimeError("No successful SISSO runs — check grid or data")

print("\nTop-3 configurations by LOOCV RMSE ↓")
print(df_results.nsmallest(3, "rmse_cv")
                 [["n_expansion","n_term","k","rmse_cv","r2_cv"]]
                 .reset_index(drop=True)
                 .to_string(index=False))

print("\nTop-3 configurations by LOOCV R² ↑")
print(df_results.nlargest(3, "r2_cv")
                 [["n_expansion","n_term","k","rmse_cv","r2_cv"]]
                 .reset_index(drop=True)
                 .to_string(index=False))

print("\nBest (min-RMSE) params :", {k: best_by_rmse[k] for k in grid})
print("  └─ LOOCV  RMSE =", best_by_rmse["rmse_cv"],
      "   R² =", best_by_rmse["r2_cv"])
print("\nBest (max-R²)  params :", {k: best_by_r2[k] for k in grid})
print("  └─ LOOCV  RMSE =", best_by_r2["rmse_cv"],
      "   R² =", best_by_r2["r2_cv"])
print("\nBest-RMSE equation:\n", best_by_rmse["equation"][:200], "…")

# Save result
elapsed = (time.perf_counter() - t_start) / 60 

out_dir  = Path("results"); out_dir.mkdir(exist_ok=True, parents=True)
stamp    = datetime.datetime.now().strftime("%Y%m%d_%H%M")
out_path = out_dir / f"sisso_loocv_weighted_{stamp}.csv"
df_results.to_csv(out_path, index=False)

print(f"\nCompleted in {elapsed:.1f} min — results saved to → {out_path}")

Grid combo → {'n_expansion': 3, 'n_term': 1, 'k': 10, 'operators': ('+', '-', '*', '/', 'exp', 'pow(-1)'), 'use_gpu': True}
Ops_match: True  k_match: n/a  NaN_coef: False
Finished:  RMSE_cv= 0.959 R2_cv=0.311  R2_full=0.337 (elapsed  14.07 s)
Grid combo → {'n_expansion': 3, 'n_term': 1, 'k': 10, 'operators': ('+', '-', '*', '/', 'exp', 'ln', 'pow(2)', 'pow(1/2)', 'pow(-1)'), 'use_gpu': True}
Ops_match: True  k_match: n/a  NaN_coef: False
Finished:  RMSE_cv= 0.959 R2_cv=0.311  R2_full=0.337 (elapsed  27.95 s)
Grid combo → {'n_expansion': 3, 'n_term': 1, 'k': 10, 'operators': ('+', '-', '*', '/', 'exp', 'ln', 'pow(2)', 'pow(3)', 'pow(1/2)', 'pow(1/3)', 'pow(-1)'), 'use_gpu': True}
Ops_match: True  k_match: n/a  NaN_coef: False
Finished:  RMSE_cv= 0.959 R2_cv=0.311  R2_full=0.337 (elapsed  46.30 s)
Grid combo → {'n_expansion': 3, 'n_term': 1, 'k': 20, 'operators': ('+', '-', '*', '/', 'exp', 'pow(-1)'), 'use_gpu': True}
Ops_match: True  k_match: n/a  NaN_coef: False
Finished:  RMSE_cv= 0.

## Descriptor Frequency Analysis based on the LOOCV result

In [10]:
def to_df(counter, name):
    return (pd.DataFrame(counter.items(), columns=[name, "count"])
              .sort_values("count", ascending=False)
              .reset_index(drop=True))

def strip_coeff(expr):
    """
    Remove only the *leading* numeric coefficient of a SymPy term,
    keeping numbers buried inside functions or exponents.
    """
    expr = sp.simplify(expr)
    if expr.is_Mul:
        non_num = [a for a in expr.args if not a.is_Number]
        return sp.Mul(*non_num) if non_num else expr
    return expr

TOP_FRAC = 0.30        # analyse TOP 30 % of models
TOP_N    = 10          
score_col = "r2_cv"    
equation_col = "equation"

real_syms   = sp.symbols(" ".join(clean_feature_list))
sym_locals  = {name: sym for name, sym in zip(clean_feature_list, real_syms)}
real_set    = set(real_syms)

df_sorted = df_results.sort_values(score_col, ascending=False)
cut_idx   = max(1, int(len(df_sorted) * TOP_FRAC))       
df_good   = df_sorted.iloc[:cut_idx]

print(f"\nAnalysing top {TOP_FRAC:.0%} "
      f"({len(df_good)}/{len(df_results)}) equations by {score_col}")

interaction_counts = Counter()
single_counts      = Counter()

for raw in df_good[equation_col]:
    eqn_str = raw.lstrip("'")
    if eqn_str.startswith("="):
        eqn_str = eqn_str.split("=", 1)[1]
    eqn_str = eqn_str.replace("^", "**")

    # Sympify into a SymPy expression tree
    expr = sp.sympify(eqn_str, locals=sym_locals)

    # Each additive term separately
    for term in expr.as_ordered_terms():
        core  = strip_coeff(term)
        feats = [s for s in core.free_symbols if s in real_set]

        if len(feats) > 1:
            interaction_counts[str(core)] += 1
        elif len(feats) == 1:
            single_counts[str(core)]      += 1



# Save results
inter_df  = to_df(interaction_counts, "interaction_term")
single_df = to_df(single_counts,      "feature_term")

print("\n=== Top interaction terms ===")
print(inter_df.head(TOP_N).to_string(index=False))

print("\n=== Top single-feature terms ===")
print(single_df.head(TOP_N).to_string(index=False))

out_dir = Path("results")          
freq_csv = out_dir / f"term_freq_{stamp}.csv"          
top_csv  = out_dir / f"term_freq_top{TOP_N}_{stamp}.csv"  

inter_df.to_csv(freq_csv.with_name(freq_csv.stem + "_inter.csv"), index=False)
single_df.to_csv(freq_csv.with_name(freq_csv.stem + "_single.csv"), index=False)

pd.concat(
    [inter_df.head(TOP_N).assign(kind="interaction"),
     single_df.head(TOP_N).assign(kind="single")], ignore_index=True
).to_csv(top_csv, index=False)

print(f"\nSaved term frequencies to:\n  • {freq_csv.with_name(freq_csv.stem + '_inter.csv')}\n  • {freq_csv.with_name(freq_csv.stem + '_single.csv')}\n  • {top_csv}")


Analysing top 30% (7/24) equations by r2_cv

=== Top interaction terms ===
               interaction_term  count
          G_time*H_2/(Ar - H_2)      7
G_temp*G_time/(G_temp + R_temp)      3
G_time*(G_temp + R_temp)/R_temp      3
       G_time*(-Size + Te)/Size      3

=== Top single-feature terms ===
Empty DataFrame
Columns: [feature_term, count]
Index: []

Saved term frequencies to:
  • results/term_freq_20250827_1300_inter.csv
  • results/term_freq_20250827_1300_single.csv
  • results/term_freq_top10_20250827_1300.csv


# Out-of-Bag (OOB) Descriptor Cross-Validation

In [11]:
X = df_model[clean_feature_list].values
y = df_model.iloc[:, 0].values

 #Select the best hyper-parameters by LOOCV R²
best_row = df_results.loc[df_results["r2_cv"].idxmax()]
best_params = {k: best_row[k] for k in ["n_expansion","n_term","k","operators"]}

print("Best LOOCV  R² :", best_row["r2_cv"])
print("Chosen params :", best_params)

# Helper functions
def oob_mask(idx_bt, n):
    mask = np.ones(n, bool);  mask[idx_bt] = False
    return mask

def pretty(eqn_raw, locals_dict):
    eqn_str = eqn_raw.lstrip("'").lstrip("=")  # drop excel prefix
    return str(sp.sympify(eqn_str.replace("^","**"), locals=locals_dict))

# Build dict for SymPy parsing
sym_loc = {**{n: sp.symbols(n) for n in clean_feature_list},
           "exp": sp.exp, "ln": sp.log, "pow": sp.Pow}

# Bootstrap ensemble 
B = 200
rng = np.random.default_rng(42)
n   = len(X)

records, eq_counter = [], Counter()

with parallel_backend("loky", inner_max_num_threads=4):
    t0 = time.perf_counter()
    for b in range(1, B + 1):
        idx_bt = rng.choice(n, n, replace=True)
        df_bt  = df_model.iloc[idx_bt]

        with silence_torchsisso():
            with pandas_safe_tensors():
                sm = SissoModel(
                        data           = df_bt,
                        dimensionality = dimensionality, 
                        output_dim     = output_dim,
                        use_gpu        = True,
                        **best_params
                    )
                rmse_fit, eqn_raw, _ = sm.fit()      

        eq_str = pretty(eqn_raw, sym_loc)
        eq_counter[eq_str] += 1

        # OOB R²
        mask = oob_mask(idx_bt, n)

        if not mask.any():   
            oob_r2 = np.nan
        else:
            try:
                # evaluate with caret-fix
                y_hat_oob, _ = sm.evaluate(eqn_raw.replace("^", "**"),
                                        df_model.iloc[mask])

                # check for numeric sanity
                if not np.isfinite(y_hat_oob).all():
                    raise ValueError("nan/inf in OOB prediction")

                oob_r2 = r2_score(y[mask], y_hat_oob)

            except Exception as e:
                print(f"[DEBUG] bootstrap {b}: OOB evaluation skipped → {e}")
                oob_r2 = np.nan    

        records.append({"boot_id": b, "equation": eq_str, "oob_r2": oob_r2})

        if b % 20 == 0 or b == B:
            print(f"[{b:3d}/{B}]  latest OOB R² = {oob_r2:5.3f}")

elapsed = (time.perf_counter() - t0) / 60
print(f"\nBootstraps finished in {elapsed:.1f} min")

# Save raw bootstrap table
out_dir = Path("results");  out_dir.mkdir(exist_ok=True, parents=True)
stamp   = datetime.datetime.now().strftime("%Y%m%d_%H%M")
csv_out = out_dir / f"sisso_bootstrap_{stamp}.csv"
pd.DataFrame(records).to_csv(csv_out, index=False)
print("Saved →", f"sisso_bootstrap_{stamp}.csv")


Best LOOCV  R² : 0.35380974267081655
Chosen params : {'n_expansion': np.int64(3), 'n_term': np.int64(3), 'k': np.int64(20), 'operators': ('+', '-', '*', '/', 'exp', 'pow(-1)')}
[ 20/200]  latest OOB R² = 0.299
[ 40/200]  latest OOB R² = 0.095
[ 60/200]  latest OOB R² = 0.186
[ 80/200]  latest OOB R² = 0.081
[100/200]  latest OOB R² = 0.043
[120/200]  latest OOB R² = -1.132
[140/200]  latest OOB R² = 0.354
[160/200]  latest OOB R² = 0.087
[180/200]  latest OOB R² = -1.384
[200/200]  latest OOB R² = 0.229

Bootstraps finished in 0.9 min
Saved → sisso_bootstrap_20250827_1304.csv


## Descriptor Frequency Analysis after OOB

In [12]:
# Whole-equation frequency
eq_df = (pd.DataFrame(eq_counter.items(), columns=["equation","count"])
           .sort_values("count", ascending=False))
print("\nTop equations across 200 bootstraps:")
print(eq_df.head(10).to_string(index=False))


# Term-level frequencies filtered by OOB R²
keep_top_frac = 0.30            # top 30 %
score_field   = "oob_r2"        

records_sorted = sorted(records, key=lambda r: r[score_field], reverse=True)
cut = max(1, int(len(records_sorted) * keep_top_frac))
high_recs = records_sorted[:cut]

print(f"\nAnalysing top {keep_top_frac:.0%} "
      f"({len(high_recs)}/{len(records)}) equations by {score_field}")

# Frequency Analysis
interaction_counter, single_counter = Counter(), Counter()
real_set = set(real_syms)

for rec in high_recs:
    expr = sp.sympify(rec["equation"], locals={s.name: s for s in real_syms})
    for term in expr.as_ordered_terms():
        core  = strip_coeff(term)
        feats = [s for s in core.free_symbols if s in real_set]

        if len(feats) > 1:
            interaction_counter[str(core)] += 1
        elif len(feats) == 1:
            single_counter[str(core)]      += 1

# Saving results
inter_df = (pd.DataFrame(interaction_counter.items(),
                         columns=["interaction_term","count"])
              .sort_values("count", ascending=False)
              .reset_index(drop=True))

single_df = (pd.DataFrame(single_counter.items(),
                          columns=["feature_term","count"])
               .sort_values("count", ascending=False)
               .reset_index(drop=True))

print(f"\nFiltered to {len(high_recs)} / {len(records)} bootstraps with "
      f"top {keep_top_frac*100:.2f}% of the equation")

print("\n=== Top single-feature terms ===")
print(single_df.head(10))

oob_stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
out_dir   = Path("results"); out_dir.mkdir(exist_ok=True, parents=True)

inter_df.to_csv(out_dir / f"oob_termfreq_inter_{oob_stamp}.csv",  index=False)
single_df.to_csv(out_dir / f"oob_termfreq_single_{oob_stamp}.csv", index=False)
print(f"\nSaved OOB term frequencies → {out_dir}")


Top equations across 200 bootstraps:
                                                                                                                                                                  equation  count
                                                                    0.03562728041553896*G_time*(-G_temp + R_temp)/R_temp + 0.6296861377506537 + 0.0302734375*G_time*H_2/Ar      1
                                                                     1.1732100043591978 + 0.0355224609375*G_time*R_temp/G_temp - 0.041227058136480005*G_time*(Ar - H_2)/Ar      1
                                                                         0.05671453809450916*G_time*Te/(Size + Te) - 0.035003662109375*G_time*Te/Size + 0.6332279860505665      1
                                                                                 0.03384861459364573*Ar*G_time/(Ar - H_2) - 0.035003662109375*G_time - 0.19521033129904108      1
                                                                        